# RSNA Knee Abnormality Detection
## State-of-the-Art Multi-Planar DINOv2 Architecture (Sagittal + Coronal + Axial)
---
### Architecture Highlights:
1. **Multi-Planar Sequence Extraction**: Dynamically selects and geometrically processes **Sagittal, Coronal, and Axial** MRI series.
2. **Physical Geometry Field-of-View Cropping**: Normalizes spatial pixel spacing to exact 130mm anatomical FOV.
3. **Cross-View Gated Attention Transformer**: Fuses 3D multi-planar volumetric features using DINOv2 backbone.
4. **Asymmetric Weighted Loss & Label Smoothing**: Addresses severe class imbalance across the 12 knee abnormalities.
5. **5-Fold Stratified Group Cross-Validation**: Prevents data leakage across patient studies.

In [ ]:
# CELL 1: Imports & Global Configuration
import os
import sys
import gc
import math
import glob
import random
import numpy as np
import pandas as pd
import cv2
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.amp
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

class CFG:
    seed = 42
    base_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
    if not os.path.exists(base_dir):
        base_dir = "/kaggle/input/rsna-knee-abnormality-detection"
        
    train_csv = os.path.join(base_dir, "train.csv")
    train_series_csv = os.path.join(base_dir, "train_series.csv")
    train_images_dir = os.path.join(base_dir, "train_series")
    
    llm_labels_dir = "/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels"
    pretrained_weights_dir = "/kaggle/input/datasets/pilkwang/rsna-knee-weights"
    
    # Medical Spatial Geometry
    crop_mm = 130.0
    image_size = 336
    num_views = 3 # Sagittal, Coronal, Axial
    slices_per_view = 3
    
    # Model & Training
    model_name = "vit_small_patch14_reg4_dinov2.lvd142m"
    n_folds = 5
    fold_to_train = 0
    epochs = 6
    batch_size = 8
    lr = 2e-4
    min_lr = 1e-6
    weight_decay = 1e-4
    num_workers = 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    target_cols = [
        "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
        "Medial OA", "Lateral OA", "PF OA", "Effusion",
        "Synovitis", "Baker's", "Contusion", "Fracture"
    ]

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.seed)
print(f"Device: {CFG.device} | PyTorch: {torch.__version__}")


In [ ]:
# CELL 2: Data Exploration & LLM Report Label Integration
train_df = pd.read_csv(CFG.train_csv)
train_series_df = pd.read_csv(CFG.train_series_csv)
print(f"Raw train studies: {len(train_df)} | Raw series entries: {len(train_series_df)}")

# Check LLM Labels
llm_csv_candidates = [
    os.path.join(CFG.llm_labels_dir, "report_labels_v2.csv"),
    os.path.join(CFG.llm_labels_dir, "train.csv"),
    os.path.join(CFG.llm_labels_dir, "train_llm_labels.csv")
]

llm_loaded = False
for cand in llm_csv_candidates:
    if os.path.exists(cand):
        print(f"Found LLM extracted labels at: {cand}")
        llm_df = pd.read_csv(cand)
        train_df.set_index("StudyInstanceUID", inplace=True)
        llm_df.set_index("StudyInstanceUID", inplace=True)
        train_df.update(llm_df)
        train_df.reset_index(inplace=True)
        llm_loaded = True
        break

if not llm_loaded:
    print("Notice: Training strictly on official provided labels.")

train_df = train_df.dropna(subset=CFG.target_cols).reset_index(drop=True)
print(f"Total usable training studies: {len(train_df)}")

# Calculate and display class prevalence
prev = train_df[CFG.target_cols].mean() * 100
plt.figure(figsize=(12, 4))
sns.barplot(x=prev.values, y=prev.index, palette="mako")
plt.title("Prevalence of 12 Knee Abnormalities (%)", fontsize=14)
plt.xlabel("Percentage Positive")
plt.show()


In [ ]:
# CELL 3: Physical Geometry DICOM Engine & Multi-Plane Selector
def get_slice_position(dcm):
    if hasattr(dcm, "ImageOrientationPatient") and hasattr(dcm, "ImagePositionPatient"):
        try:
            pos = np.array([float(x) for x in dcm.ImagePositionPatient])
            ori = np.array([float(x) for x in dcm.ImageOrientationPatient])
            normal = np.cross(ori[0:3], ori[3:6])
            return np.dot(pos, normal)
        except:
            pass
    if hasattr(dcm, "SliceLocation"): return float(dcm.SliceLocation)
    if hasattr(dcm, "InstanceNumber"): return float(dcm.InstanceNumber)
    return 0.0

def sort_dicom_files(dicom_paths):
    metadata = []
    for path in dicom_paths:
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True)
            metadata.append({
                "path": path,
                "pos": get_slice_position(dcm),
                "instance": getattr(dcm, "InstanceNumber", 0)
            })
        except:
            continue
    metadata.sort(key=lambda x: (x["pos"], x["instance"], x["path"]))
    return [x["path"] for x in metadata]

def read_and_crop_dicom(path, crop_mm=130.0, target_size=336):
    try:
        dcm = pydicom.dcmread(path)
        img = apply_voi_lut(dcm.pixel_array, dcm)
        if dcm.PhotometricInterpretation == "MONOCHROME1":
            img = np.amax(img) - img
        img = img - np.min(img)
        img = img / (np.max(img) + 1e-6)
        img = (img * 255).astype(np.uint8)
        
        spacing_y, spacing_x = 1.0, 1.0
        if hasattr(dcm, "PixelSpacing"):
            try: spacing_y, spacing_x = [float(s) for s in dcm.PixelSpacing]
            except: pass
        
        h, w = img.shape
        crop_h = min(int(crop_mm / spacing_y), h)
        crop_w = min(int(crop_mm / spacing_x), w)
        start_y = (h - crop_h) // 2
        start_x = (w - crop_w) // 2
        
        cropped = img[start_y:start_y+crop_h, start_x:start_x+crop_w]
        return cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_CUBIC)
    except:
        return np.zeros((target_size, target_size), dtype=np.uint8)

def extract_plane_triplets(series_dir, crop_mm=130.0, target_size=336):
    if not series_dir or not os.path.exists(series_dir):
        return np.zeros((3, target_size, target_size), dtype=np.uint8)
        
    dcm_files = [os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith(".dcm")]
    if not dcm_files:
        return np.zeros((3, target_size, target_size), dtype=np.uint8)
        
    sorted_files = sort_dicom_files(dcm_files)
    n = len(sorted_files)
    if n >= 3:
        # Take center 3 slices
        mid = n // 2
        f_r = sorted_files[mid - 1]
        f_g = sorted_files[mid]
        f_b = sorted_files[min(n - 1, mid + 1)]
    elif n == 2:
        f_r, f_g, f_b = sorted_files[0], sorted_files[1], sorted_files[1]
    else:
        f_r = f_g = f_b = sorted_files[0]
        
    img_r = read_and_crop_dicom(f_r, crop_mm, target_size)
    img_g = read_and_crop_dicom(f_g, crop_mm, target_size)
    img_b = read_and_crop_dicom(f_b, crop_mm, target_size)
    
    return np.stack([img_r, img_g, img_b], axis=-1)


In [ ]:
# CELL 4: Multi-Planar Dataset & Advanced Medical Augmentations
train_transforms = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.08, rotate_limit=12, p=0.6, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.CoarseDropout(max_holes=6, max_height=24, max_width=24, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class RSNAKneeMultiViewDataset(Dataset):
    def __init__(self, df, series_df, images_base_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.images_base_dir = images_base_dir
        self.is_train = is_train
        self.transform = train_transforms if is_train else valid_transforms
        self.target_cols = CFG.target_cols
        self.planes = ["Sagittal", "Coronal", "Axial"]

    def __len__(self):
        return len(self.df)

    def _get_series_for_plane(self, study_series, plane):
        plane_series = study_series[study_series["Anatomical_Plane"] == plane]
        if len(plane_series) == 0:
            return None
        # Prioritize Fluid Sensitive or Fat Suppressed acquisitions
        fs_series = plane_series[plane_series["Fluid_Sensitive"] == 1]
        if len(fs_series) > 0:
            return str(fs_series.iloc[0]["SeriesInstanceUID"])
        return str(plane_series.iloc[0]["SeriesInstanceUID"])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = str(row["StudyInstanceUID"])
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == study_id]
        
        view_tensors = []
        for plane in self.planes:
            series_id = self._get_series_for_plane(study_series, plane)
            series_dir = None
            if series_id:
                series_dir = os.path.join(self.images_base_dir, study_id, series_id)
                
            rgb_img = extract_plane_triplets(series_dir, CFG.crop_mm, CFG.image_size)
            transformed = self.transform(image=rgb_img)["image"] # (3, H, W)
            view_tensors.append(transformed)
            
        # Stack views -> Shape: (3, 3, H, W) -> (Views, Channels, Height, Width)
        stacked_views = torch.stack(view_tensors, dim=0)
        
        if all(col in self.df.columns for col in self.target_cols):
            targets = torch.tensor(row[self.target_cols].values.astype(np.float32))
            return stacked_views, targets
        return stacked_views, study_id


In [ ]:
# CELL 5: Multi-Planar Vision Transformer Model
class MultiViewDINOv2KneeModel(nn.Module):
    def __init__(self, model_name=CFG.model_name, num_classes=12, num_views=3, pretrained=True):
        super().__init__()
        self.num_views = num_views
        self.encoder = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            dynamic_img_size=True
        )
        embed_dim = self.encoder.num_features
        
        # Cross-View Gated Attention Pooling
        self.view_gate = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Multi-Label Classifier Head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, num_views, C, H, W)
        B, V, C, H, W = x.shape
        x_flat = x.view(B * V, C, H, W)
        
        # Extract features per plane: (B * V, embed_dim)
        features = self.encoder(x_flat)
        features = features.view(B, V, -1) # (B, V, embed_dim)
        
        # Attention scores across planes: (B, V, 1)
        attn_scores = self.view_gate(features)
        attn_weights = torch.softmax(attn_scores, dim=1)
        
        # Weighted multi-plane feature: (B, embed_dim)
        pooled_features = torch.sum(features * attn_weights, dim=1)
        
        # 12 Abnormality Predictions: (B, 12)
        logits = self.head(pooled_features)
        return logits

test_model = MultiViewDINOv2KneeModel(pretrained=False).to(CFG.device)
sample_input = torch.randn(2, 3, 3, 336, 336).to(CFG.device)
sample_out = test_model(sample_input)
print(f"Multi-View Architecture initialized! Test output shape: {sample_out.shape}")


In [ ]:
# CELL 6: Asymmetric Loss, Metric & Epoch Training Engine
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        # x: logits, y: targets (0 or 1, or soft probabilities)
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos
        
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)
            
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos * ((1.0 - xs_pos) ** self.gamma_pos) + los_neg * (xs_pos ** self.gamma_neg)
        return -loss.sum(dim=-1).mean()

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, scheduler, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(dataloader, desc="Training")
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    scheduler.step()
    return running_loss / len(dataloader.dataset)

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_targets, all_preds = [], []
    
    for images, targets in tqdm(dataloader, desc="Validation"):
        images, targets = images.to(device), targets.to(device)
        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)
            
        running_loss += loss.item() * images.size(0)
        all_targets.append(targets.cpu().numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        
    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)
    
    # Binarize targets for metric calculation
    binary_targets = (all_targets >= 0.5).astype(int)
    auc_scores = []
    for i in range(12):
        if len(np.unique(binary_targets[:, i])) > 1:
            auc = roc_auc_score(binary_targets[:, i], all_preds[:, i])
            auc_scores.append(auc)
            
    mean_auc = np.mean(auc_scores) if auc_scores else 0.0
    return running_loss / len(dataloader.dataset), mean_auc, all_preds


In [ ]:
# CELL 7: 5-Fold Cross-Validation Training Pipeline
kf = KFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df["fold"] = -1
for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    train_df.loc[val_idx, "fold"] = fold

print(f"\nFold Distribution:\n{train_df['fold'].value_counts().sort_index()}")

# Prepare Training Split for designated fold
train_split = train_df[train_df["fold"] != CFG.fold_to_train].reset_index(drop=True)
val_split = train_df[train_df["fold"] == CFG.fold_to_train].reset_index(drop=True)
print(f"\nFold {CFG.fold_to_train}: {len(train_split)} Train Studies | {len(val_split)} Val Studies")

train_loader = DataLoader(
    RSNAKneeMultiViewDataset(train_split, train_series_df, CFG.train_images_dir, is_train=True),
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True
)
val_loader = DataLoader(
    RSNAKneeMultiViewDataset(val_split, train_series_df, CFG.train_images_dir, is_train=False),
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True
)

# Initialize Multi-Planar Model
model = MultiViewDINOv2KneeModel(model_name=CFG.model_name, num_classes=12, pretrained=True).to(CFG.device)

# Load pretrained backbone if available
if os.path.exists(CFG.pretrained_weights_dir):
    pt_files = sorted(glob.glob(os.path.join(CFG.pretrained_weights_dir, "*.pt")))
    if pt_files:
        chosen_pt = pt_files[CFG.fold_to_train % len(pt_files)]
        print(f"Loading pretrained backbone weights from: {chosen_pt}")
        try:
            st = torch.load(chosen_pt, map_location=CFG.device)
            model.load_state_dict(st, strict=False)
            print("Pretrained backbone loaded successfully!")
        except Exception as e:
            print(f"Notice: {e}")

criterion = AsymmetricLoss(gamma_neg=2, gamma_pos=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=CFG.min_lr)
scaler = torch.amp.GradScaler("cuda")

best_auc = 0.0
save_name = f"multiview_dinov2_fold{CFG.fold_to_train}_best.pth"

print(f"\n========== STARTING FOLD {CFG.fold_to_train} TRAINING ({CFG.epochs} EPOCHS) ==========")
for epoch in range(CFG.epochs):
    print(f"\n--- Epoch {epoch+1}/{CFG.epochs} ---")
    t_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler, CFG.device)
    v_loss, v_auc, _ = evaluate(model, val_loader, criterion, CFG.device)
    
    print(f"Epoch {epoch+1} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Macro AUC: {v_auc:.4f}")
    
    if v_auc > best_auc:
        best_auc = v_auc
        torch.save(model.state_dict(), save_name)
        print(f"🏆 Saved new best checkpoint: {save_name} (AUC: {best_auc:.4f})")

print(f"\nFold {CFG.fold_to_train} Complete! Best Validation Macro AUC: {best_auc:.4f}")
